In [1]:
# Clone of test-1PA-traj.ipynb, modified to trigger some warnings
%load_ext autoreload
%autoreload 2

In [2]:
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.IPAT1R import Trajectory1PAT1R
import matplotlib.pyplot as plt
import numpy as np

# ODE

In [3]:
ode = Trajectory1PAT1R()

In [4]:
m1 = 1e6       # primary mass [solar masses]
m2 = 1e6       # secondary mass [solar masses]

a  = -0.2       # primary spin chi1 (dimensionless)
chit2  = -0.1   # secondary spin
deltaM = 0.0

In [5]:
# Omit chi2: trigger a warning
ode.add_fixed_parameters(m1, m2, a, [])

WARNING (Trajectory1PAT1R.add_fixed_parameters): Empty list of additional arguments received in Trajectory1PAT1R. chi2 set to 0 and evolve_primary set to True by default


In [6]:
# Excess arguments: trigger a warning
ode.add_fixed_parameters(m1, m2, a, [chit2, chit2, chit2])

WARNING (Trajectory1PAT1R.add_fixed_parameters): Only 2 additional arguments (chi2, evolve_primary) expected in Trajectory1PAT1R but 3 received. Excess arguments will be ignored.


In [7]:
# Add a valid set of fixed parameters
ode.add_fixed_parameters(m1, m2, a, [chit2])

In [ ]:
# Evaluate at invalid p
try:
    ode.evaluate_rhs([4, 0.0, 1.0, 0.0, 0.0, 0.0, deltaM, 0])
except Exception as error:
    print(error)

    # FIXME: why doesn't this give an error anymore

In [ ]:
# Evaluate at invalid delta_chit1:
try:
    ode.evaluate_rhs([7.0, 0.0, 1.0, 0.0, 0.0, 0.0, deltaM, -0.1])
except Exception as error:
    print(error)

    # FIXME: why doesn't this give an error anymore

In [ ]:
# Now evaluate at valid parameters (no warning expected)
r_values = np.linspace(6.0, 15.0, 1000); delta_chit1 = 0.0;
rdot_values = [ode.evaluate_rhs([r, 0.0, 1.0, 0.0, 0.0, 0.0, deltaM, delta_chit1])[0] for r in r_values]
chit1dot_values = [ode.evaluate_rhs([r, 0.0, 1.0, 0.0, 0.0, 0.0, deltaM, delta_chit1])[-1] for r in r_values]
deltaMdot_values = [ode.evaluate_rhs([r, 0.0, 1.0, 0.0, 0.0, 0.0, deltaM, delta_chit1])[-2] for r in r_values]

# Trajectory

In [13]:
traj = EMRIInspiral(func=Trajectory1PAT1R)

In [14]:
m1 = 1e6       # primary mass [solar masses]
m2 = 1e6       # secondary mass [solar masses]

a  = -0.2       # primary spin chi1 (dimensionless)
chit2  = -0.1   # secondary spin
p0 = 15.0      # initial orbital separation [M]

dt = 1.   # time step [s]
T  = 1    # total duration [yr]

In [15]:
# Try to call trajectory with invalid value of p0:
try: 
    t, p, e, x, Phi_phi, Phi_theta, Phi_r, deltaM, deltachit1 = traj(
    m1, m2,
    a,        # chi1 (primary spin)
    5.9,
    0.0,      # e0: circular
    1.0,      # xI: equatorial prograde
    chit2,
    T=T,
    dt=dt,
    in_coordinate_time=False
    )
except Exception as error:
    print(error)

Interpolation: p out of bounds. Must be between 6.639040203567554 and 30.0.


In [16]:
# Try to call trajectory with invalid value of chi1:
try: 
    t, p, e, x, Phi_phi, Phi_theta, Phi_r, deltaM, deltachit1 = traj(
    m1, m2,
    0.3,        # chi1 (primary spin)
    p0,
    0.0,      # e0: circular
    1.0,      # xI: equatorial prograde
    chit2,
    T=T,
    dt=dt,
    in_coordinate_time=False
    )
except Exception as error:
    print(error)

Domain validity: chi1 = 0.3 out of bounds. Must be between -0.2 and 0.2.


In [ ]:
# Call with a valid set of parameters (no error expected)
t, p, e, x, Phi_phi, Phi_theta, Phi_r, deltaM, deltachit1 = traj(
    m1, m2,
    a,        # chi1 (primary spin)
    p0,
    0.0,      # e0: circular
    1.0,      # xI: equatorial prograde
    chit2,
    T=T,
    dt=dt,
    in_coordinate_time=False
)